In [6]:
import pandas as pd
# โหลดข้อมูลที่คลีนเสร็จแล้วทั้ง 5 ไฟล์
df_pm25 = pd.read_csv('Cleaned_PM25.csv')
df_rain = pd.read_csv('Cleaned_Rainfall.csv')
df_veh  = pd.read_csv('Cleaned_Vehicles.csv')
df_pop  = pd.read_csv('Cleaned_Population.csv')
df_fac  = pd.read_csv('Cleaned_Factories.csv')

# ปรับมาตรฐานชื่อคอลัมน์และชนิดข้อมูล
# ปัญหา: ไฟล์ทั้ง 5 มาจากคนละแหล่ง อาจมีชื่อคอลัมน์พิมพ์เล็ก/ใหญ่ หรือมีช่องว่างซ่อนอยู่ (นำไปสู่ KeyError)
# วิธีแก้: เขียนลูปกวาดล้างและปรับฟอร์แมต (Format) ของทุกตารางให้เป็นมาตรฐานเดียวกันเป๊ะๆ
dfs = [df_pm25, df_rain, df_veh, df_pop, df_fac]
for i, df in enumerate(dfs):
    # ลบช่องว่าง (Whitespace) ที่อาจซ่อนอยู่หน้า/หลังชื่อคอลัมน์
    df.columns = df.columns.str.strip()

    # สร้าง Dictionary เพื่อ Mapping ชื่อคอลัมน์ที่สะกดต่างกัน ให้กลายเป็นคำเดียวกัน
    rename_map = {
        'จังหวัด': 'Province', 'province': 'Province', 'PROVINCE': 'Province',
        'ปี': 'Year', 'year': 'Year', 'YEAR': 'Year',
        'เดือน': 'Month', 'month': 'Month', 'MONTH': 'Month'}
    df.rename(columns=rename_map, inplace=True)

    # บังคับชนิดข้อมูล (Data Casting) ให้ตรงกัน เพื่อให้คำสั่ง Join ทำงานได้ไม่มีสะดุด
    df['Year'] = df['Year'].astype(int)
    df['Month'] = df['Month'].astype(str).str.strip()
    df['Province'] = df['Province'].astype(str).str.strip()

# เริ่มกระบวนการประกอบร่าง (Data Merging / Left Join)
# เทคนิคสำคัญ: เลือกใช้ Left Join โดยยึดตาราง PM2.5 เป็น "ตารางหลัก"
# เหตุผล: เพราะ PM2.5 คือตัวแปรตาม (Target Variable) ที่เราต้องการทำนาย ถ้าเดือนไหนไม่มีค่าฝุ่น
# ข้อมูลรถหรือประชากรในเดือนนั้นก็ไม่มีประโยชน์ที่จะนำมาเทรนโมเดล
df_final = pd.merge(df_pm25, df_veh, on=['Year', 'Month', 'Province'], how='left')
df_final = pd.merge(df_final, df_rain, on=['Year', 'Month', 'Province'], how='left')
df_final = pd.merge(df_final, df_pop, on=['Year', 'Month', 'Province'], how='left')
df_final = pd.merge(df_final, df_fac, on=['Year', 'Month', 'Province'], how='left')

# จัดระเบียบคอลัมน์และตั้งชื่อใหม่ (Feature Reordering)
# เขียนโค้ดแบบ Dynamic ค้นหาคอลัมน์ PM2.5 (เผื่อไฟล์ต้นฉบับตั้งชื่อต่างกันเช่น pm25 หรือ PM 2.5)
pm25_col = [c for c in df_final.columns if 'pm2' in c.lower() or 'pm 2' in c.lower()][0]

# จัดเรียงคอลัมน์ให้สวยงาม (เอาตัวแปรอิสระขึ้นก่อน แล้วปิดท้ายด้วยตัวแปรตาม)
df_final = df_final[['Year', 'Month', 'Province', 'Total_Population', 'Total_Vehicles', 'New_Factories', 'Rainfall', pm25_col]]
df_final.rename(columns={pm25_col: 'PM2.5'}, inplace=True)

# ตัดแถวที่ค่าฝุ่นหายไปจริงๆ ทิ้ง (Drop Missing Target) เพื่อไม่ให้โมเดลสับสน
df_final = df_final.dropna(subset=['PM2.5'])

# เติม 0 ให้กับข้อมูลโรงงานที่อาจไม่มีข้อมูลหลังจากการ Join
# ความหมายคือ: ถ้า Join ไม่เจอ แปลว่าเดือนนั้น/จังหวัดนั้น ไม่มีการตั้งโรงงานใหม่เลย (คือ 0 โรง)
df_final['New_Factories'] = df_final['New_Factories'].fillna(0)

# บันทึก  Dataset
df_final.to_csv('Final_AirQuality_Dataset.csv', index=False, encoding='utf-8-sig')

print(f"ได้ Dataset จำนวน {len(df_final)} แถว")
display(df_final.head(10))

ได้ Dataset จำนวน 4620 แถว


,Year,Month,Province,Total_Population,Total_Vehicles,New_Factories,Rainfall,PM2.5
0,2020,Jan,กรุงเทพมหานคร,5588222,10971799.0,64,NaN,40.583333
1,2020,Jan,นนทบุรี,1276745,189955.0,35,NaN,42.000000
2,2020,Jan,ปทุมธานี,1176412,175071.0,77,NaN,41.000000
3,2020,Jan,สมุทรปราการ,1351479,162670.0,205,NaN,40.600000
4,2020,Jan,สมุทรสาคร,586199,244894.0,288,NaN,44.000000
5,2020,Jan,นครปฐม,920729,504023.0,116,NaN,42.000000
6,2020,Jan,พระนครศรีอยุธยา,819088,496186.0,55,NaN,49.000000
7,2020,Jan,สระบุรี,643828,448216.0,56,NaN,58.000000
8,2020,Jan,ลพบุรี,742928,440032.0,27,NaN,37.500000
9,2020,Jan,สิงห์บุรี,205898,141577.0,6,NaN,42.000000


In [12]:
import pandas as pd
import numpy as np

# 1. โหลดข้อมูลต้นฉบับ
# แนะนำให้ใช้ไฟล์ Final_AirQuality_Dataset.csv (ตัวที่ยังมีค่า NaN) เพื่อความแม่นยำในการเลือกจุดที่จะเติม
df = pd.read_csv('Final_AirQuality_Dataset.csv')

# 2. คำนวณค่าเฉลี่ยปริมาณน้ำฝนรายจังหวัดและรายเดือน จากปี 2021-2024
# เราจะกรองเอาข้อมูลเฉพาะปี 2021-2024 มาหาค่าเฉลี่ย
df_recent = df[df['Year'].isin([2021, 2022, 2023, 2024])].copy()
rain_baseline = df_recent.groupby(['Province', 'Month'])['Rainfall'].mean().reset_index()
rain_baseline.rename(columns={'Rainfall': 'Rainfall_Avg_21_24'}, inplace=True)

# 3. นำค่าเฉลี่ยที่คำนวณได้มาเชื่อม (Merge) กลับเข้าสู่ตารางหลัก
df = df.merge(rain_baseline, on=['Province', 'Month'], how='left')

# 4. เติมค่าว่าง (Imputation) เฉพาะปี 2020 ด้วยค่าเฉลี่ยที่หามาได้
# เงื่อนไข: ถ้าเป็นปี 2020 และค่า Rainfall เดิมเป็นค่าว่าง ให้แทนที่ด้วย Rainfall_Avg_21_24
mask_2020_missing = (df['Year'] == 2020) & (df['Rainfall'].isna())
df.loc[mask_2020_missing, 'Rainfall'] = df.loc[mask_2020_missing, 'Rainfall_Avg_21_24']

# 5. ลบคอลัมน์ชั่วคราวทิ้ง
df.drop(columns=['Rainfall_Avg_21_24'], inplace=True)

# 6. เติมค่าว่างส่วนที่เหลือ (เช่น ถ้าปี 21-24 ก็ไม่มีข้อมูลของจังหวัดนั้นในเดือนนั้นด้วย)
# ให้เติมด้วย 0 (ตามหลักการที่ว่าถ้าไม่มีบันทึกคือฝนไม่ตก)
df['Rainfall'] = df['Rainfall'].fillna(0)

# 7. จัดการข้อมูลส่วนอื่นๆ (ประชากรและรถยนต์) ตามหลักการเดิมในงานวิจัย
# เติมด้วยค่าเฉลี่ยของจังหวัดนั้นๆ
df['Total_Vehicles'] = df.groupby('Province')['Total_Vehicles'].transform(lambda x: x.fillna(x.mean()))
df['Total_Population'] = df.groupby('Province')['Total_Population'].transform(lambda x: x.fillna(x.mean()))

# 8. อุดช่องโหว่สุดท้ายด้วยค่าเฉลี่ยทั้งประเทศ (ถ้ายังมีค่าว่างเหลืออยู่)
df = df.fillna(df.mean(numeric_only=True))

# 9. ตรวจสอบข้อมูลปี 2020 หลังการแก้ไข
print("--- ตรวจสอบข้อมูลปี 2020 หลังเติมค่าเฉลี่ยปี 21-24 ---")
display(df[df['Year'] == 2020].head(10))

# 10. ตรวจสอบค่าว่างคงเหลือทั้งหมด
print("\n--- ตรวจสอบค่าว่างคงเหลือทั้งหมด ---")
print(df.isnull().sum())

# 11. บันทึกไฟล์ใหม่
df.to_csv('AirQuality_Final.csv', index=False, encoding='utf-8-sig')

--- ตรวจสอบข้อมูลปี 2020 หลังเติมค่าเฉลี่ยปี 21-24 ---


,Year,Month,Province,Total_Population,Total_Vehicles,New_Factories,Rainfall,PM2.5
0,2020,Jan,กรุงเทพมหานคร,5588222,10971799.0,64,9.766667,40.583333
1,2020,Jan,นนทบุรี,1276745,189955.0,35,0.000000,42.000000
2,2020,Jan,ปทุมธานี,1176412,175071.0,77,0.000000,41.000000
3,2020,Jan,สมุทรปราการ,1351479,162670.0,205,0.000000,40.600000
4,2020,Jan,สมุทรสาคร,586199,244894.0,288,0.000000,44.000000
5,2020,Jan,นครปฐม,920729,504023.0,116,0.000000,42.000000
6,2020,Jan,พระนครศรีอยุธยา,819088,496186.0,55,0.000000,49.000000
7,2020,Jan,สระบุรี,643828,448216.0,56,0.000000,58.000000
8,2020,Jan,ลพบุรี,742928,440032.0,27,16.716667,37.500000
9,2020,Jan,สิงห์บุรี,205898,141577.0,6,0.000000,42.000000



--- ตรวจสอบค่าว่างคงเหลือทั้งหมด ---
Year                0
Month               0
Province            0
Total_Population    0
Total_Vehicles      0
New_Factories       0
Rainfall            0
PM2.5               0
dtype: int64
